# Evaluate Vaani-FastConformer-Telugu on IndicVoices Telugu

**Model :** `ARTPARK-IISc/Vaani-FastConformer-Telugu`  
**Dataset:** `ai4bharat/IndicVoices` — Telugu / valid split  
**Platform:** Kaggle (GPU P100)  

### Features
- ✅ Batch inference with progress bar
- ✅ **Checkpointing** — saves every N batches; resumes from last checkpoint if re-run
- ✅ WER & CER via `jiwer`
- ✅ Per-sample predictions saved to `/kaggle/working/`

### Before running
1. **Settings → Internet → On** (required to pull model + dataset from HuggingFace)
2. Add your HF token as a Kaggle Secret:
   - *Add-ons → Secrets → + New Secret*
   - Name: `HF_TOKEN`, Value: your token from https://huggingface.co/settings/tokens
3. Set accelerator to **GPU T4X2** via *Session options → Accelerator*

## Cell 1 — Install Dependencies

In [ ]:
%%capture
# NeMo + ASR extras (~4-5 min first run; cached on re-run)
!pip install Cython
!pip install 'nemo_toolkit[asr]'
!pip install datasets huggingface_hub jiwer soundfile librosa
print('Dependencies installed ✅')

## Cell 2 — HuggingFace Authentication (via Kaggle Secret)

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
hf_token = secret.get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace login successful ✅')

HuggingFace login successful ✅


## Cell 3 — Configuration

In [3]:
import os

# ── Model & Dataset ──────────────────────────────────────
MODEL_ID        = 'ARTPARK-IISc/Vaani-FastConformer-Telugu'
DATASET_ID      = 'ai4bharat/IndicVoices'
LANG_CONFIG     = 'telugu'   # IndicVoices config name (lowercase)
SPLIT           = 'valid'    # IndicVoices Telugu has 'valid' not 'test'
TARGET_SR       = 16000     # NeMo expects 16 kHz

# ── Inference ────────────────────────────────────────────
BATCH_SIZE      = 8          # lower to 4 if OOM on P100
MAX_SAMPLES     = None       # set e.g. 200 for a quick smoke-test; None = full set

# ── Checkpointing ────────────────────────────────────────
CHECKPOINT_EVERY = 50        # save checkpoint every N batches
OUTPUT_DIR       = '/kaggle/working/eval_results'
CHECKPOINT_FILE  = os.path.join(OUTPUT_DIR, 'checkpoint.json')
PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, 'predictions.jsonl')
METRICS_FILE     = os.path.join(OUTPUT_DIR, 'metrics.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready ✅')
print(f'  Output dir      : {OUTPUT_DIR}')
print(f'  Checkpoint every: {CHECKPOINT_EVERY} batches')

Config ready ✅
  Output dir      : /kaggle/working/eval_results
  Checkpoint every: 50 batches


## Cell 4 — Load Model

In [4]:
import torch
from nemo.collections.asr.models import EncDecRNNTBPEModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

model = EncDecRNNTBPEModel.from_pretrained(model_name=MODEL_ID)
model = model.to(device)
model.eval()
print('Model loaded ✅')

[NeMo W 2026-06-15 07:35:33 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-06-15 07:35:37 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-06-15 07:35:37 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-06-15 07:35:37 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.

Device: cuda
GPU   : Tesla T4


Vaani-FastConformer-Telugu.nemo:   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[NeMo I 2026-06-15 07:35:59 mixins:184] Tokenizer SentencePieceTokenizer initialized with 800 tokens


[NeMo W 2026-06-15 07:35:59 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /home/sujith/asrTraining/finetuningDataset/manifest_train_telugu.json
    sample_rate: 16000
    use_start_end_token: false
    batch_size: 16
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 40
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-06-15 07:35:59 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /home/sujith/asrTraining/finetuningData

[NeMo I 2026-06-15 07:36:02 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-06-15 07:36:03 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-06-15 07:36:03 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-06-15 07:36:05 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--ARTPARK-IISc--Vaani-FastConformer-Telugu/snapshots/f1fd11d71a55fad6d9c0f0e1bbe4ff638f44cc51/Vaani-FastConformer-Telugu.nemo.
Model loaded ✅


## Cell 5 — Load IndicVoices Telugu Dataset

In [5]:
from datasets import load_dataset

print(f'Loading {DATASET_ID} [{LANG_CONFIG}] split={SPLIT} ...')
dataset = load_dataset(DATASET_ID, LANG_CONFIG, split=SPLIT)

if MAX_SAMPLES:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f'Total samples : {len(dataset)}')
print(f'Columns       : {dataset.column_names}')

# Inspect first sample
s0 = dataset[0]
print('\nFirst sample preview:')
for k, v in s0.items():
    print(f'  {k}: {str(v)[:100]}')

Loading ai4bharat/IndicVoices [telugu] split=valid ...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

telugu/valid-00000-of-00001.parquet:   0%|          | 0.00/345M [00:00<?, ?B/s]

...

telugu/train-00059-of-00061.parquet:   0%|          | 0.00/678M [00:00<?, ?B/s]

telugu/train-00060-of-00061.parquet:   0%|          | 0.00/740M [00:00<?, ?B/s]

Generating valid split:   0%|          | 0/3295 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/269306 [00:00<?, ? examples/s]

Total samples : 3295
Columns       : ['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized']

First sample preview:
  audio_filepath: <datasets.features._torchcodec.AudioDecoder object at 0x7b319e4978c0>
  text: అమెరికా ఇంగ్లాండు ఇండియా రష్యా జపాన్
  duration: 4.649
  lang: te
  samples: 74384
  verbatim: అమెరికా ఇంగ్లాండు ఇండియా రష్యా జపాన్
  normalized: అమెరికా ఇంగ్లాండు ఇండియా రష్యా జపాన్
  speaker_id: S4257866500399061
  scenario: Extempore
  task_name: Task of Fives
  gender: Male
  age_group: 30-45
  job_type: White Collar
  qualification: Post Grad + PhD
  area: Rural
  district: Karimnagar
  state: Telangana
  occupation: Teacher
  verification_report: {'decision': 'excellent', 'low_volume': False, 'noise_intermittent': False, 'chatter_intermitten

## Cell 6 — Helper Functions

In [9]:
import numpy as np
import librosa
import soundfile as sf
import tempfile, uuid, json

# ── Audio ────────────────────────────────────────────────
def get_audio_array(sample) -> np.ndarray:
    audio = sample['audio_filepath']
    arr   = np.array(audio['array'], dtype=np.float32)
    sr    = audio['sampling_rate']
    if sr != TARGET_SR:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR)
    return arr

# ── Reference transcript ─────────────────────────────────
def get_reference(sample) -> str:
    for key in ('text', 'transcript', 'sentence', 'normalized_text'):
        if key in sample and sample[key]:
            return sample[key].strip()
    raise KeyError(f'No transcript column found in: {list(sample.keys())}')

# ── Batch transcription ──────────────────────────────────
def transcribe_batch(samples):
    tmp_paths = []
    for s in samples:
        arr  = get_audio_array(s)
        path = os.path.join(tempfile.gettempdir(), f'{uuid.uuid4().hex}.wav')
        sf.write(path, arr, TARGET_SR)
        tmp_paths.append(path)
    try:
        hyps = model.transcribe(tmp_paths, return_hypotheses=True,
                                batch_size=len(tmp_paths))
        return [h.text if hasattr(h, 'text') else str(h) for h in hyps]
    finally:
        for p in tmp_paths:
            if os.path.exists(p):
                os.remove(p)

# ── Checkpoint I/O ───────────────────────────────────────
def save_checkpoint(batch_idx, results):
    """Save progress so inference can be resumed after interruption."""
    ckpt = {'next_batch': batch_idx + 1, 'num_done': len(results)}
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f)
    # Append new predictions to JSONL (file opened in append mode)

def load_checkpoint():
    """Returns (start_batch, already_done_count) or (0, 0) if no checkpoint."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            ckpt = json.load(f)
        print(f'🔁 Resuming from batch {ckpt["next_batch"]}  '
              f'({ckpt["num_done"]} samples already done)')
        return ckpt['next_batch'], ckpt['num_done']
    return 0, 0

def load_existing_predictions():
    """Read predictions already written to the JSONL file."""
    results = []
    if os.path.exists(PREDICTIONS_FILE):
        with open(PREDICTIONS_FILE) as f:
            for line in f:
                line = line.strip()
                if line:
                    results.append(json.loads(line))
    return results

print('Helpers defined ✅')

Helpers defined ✅


## Cell 7 — Run Inference with Checkpointing

If this cell is interrupted and re-run, it automatically resumes from where it left off.

In [10]:
from tqdm.notebook import tqdm

# ── Resume or start fresh ────────────────────────────────
start_batch, num_done = load_checkpoint()
sample_log = load_existing_predictions()   # already-saved entries

all_batches = list(range(0, len(dataset), BATCH_SIZE))
remaining   = all_batches[start_batch:]

print(f'Total batches : {len(all_batches)}')
print(f'Remaining     : {len(remaining)}')
print(f'Already done  : {num_done} samples')
print()

# ── Open predictions file in append mode ─────────────────
pred_file = open(PREDICTIONS_FILE, 'a', encoding='utf-8')

try:
    for batch_num, i in enumerate(tqdm(remaining, desc='Batches')):
        global_batch_idx = start_batch + batch_num

        batch = [dataset[j] for j in range(i, min(i + BATCH_SIZE, len(dataset)))]
        preds = transcribe_batch(batch)

        for s, pred in zip(batch, preds):
            ref   = get_reference(s)
            entry = {'reference': ref, 'hypothesis': pred}
            sample_log.append(entry)
            pred_file.write(json.dumps(entry, ensure_ascii=False) + '\n')

        pred_file.flush()   # flush after every batch

        # ── Checkpoint every N batches ────────────────────
        if (batch_num + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(global_batch_idx, sample_log)
            print(f'  💾 Checkpoint saved at batch {global_batch_idx+1}  '
                  f'({len(sample_log)} samples done)')

    # Final checkpoint
    save_checkpoint(len(all_batches) - 1, sample_log)
    print(f'\n✅ Inference complete — {len(sample_log)} samples processed')

finally:
    pred_file.close()

Total batches : 412
Remaining     : 412
Already done  : 0 samples



Batches:   0%|          | 0/412 [00:00<?, ?it/s]

[NeMo W 2026-06-15 08:07:47 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:07:47 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:20, 20.52s/it]
[NeMo W 2026-06-15 08:08:07 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:08:07 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impact

  💾 Checkpoint saved at batch 50  (400 samples done)



Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  1.83it/s]
[NeMo W 2026-06-15 08:08:31 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:08:31 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  2.29it/s]
[NeMo W 2026-06-15 08:08:31 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:08:31 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cau

...

  💾 Checkpoint saved at batch 350  (2800 samples done)



Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  1.79it/s]
[NeMo W 2026-06-15 08:11:18 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:11:18 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  3.03it/s]
[NeMo W 2026-06-15 08:11:18 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:11:18 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cau

  💾 Checkpoint saved at batch 400  (3200 samples done)



Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  5.08it/s]
[NeMo W 2026-06-15 08:11:54 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:11:54 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  2.81it/s]
[NeMo W 2026-06-15 08:11:54 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 08:11:54 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cau


✅ Inference complete — 3295 samples processed


## Cell 8 — Compute WER & CER

In [11]:
from jiwer import wer, cer

references  = [e['reference']  for e in sample_log]
hypotheses  = [e['hypothesis'] for e in sample_log]

word_error_rate = wer(references, hypotheses)
char_error_rate = cer(references, hypotheses)

print('=' * 55)
print(f'  Model   : {MODEL_ID}')
print(f'  Dataset : {DATASET_ID} / {LANG_CONFIG} / {SPLIT}')
print(f'  Samples : {len(references)}')
print(f'  WER     : {word_error_rate*100:.2f}%')
print(f'  CER     : {char_error_rate*100:.2f}%')
print('=' * 55)

  Model   : ARTPARK-IISc/Vaani-FastConformer-Telugu
  Dataset : ai4bharat/IndicVoices / telugu / valid
  Samples : 3295
  WER     : 27.31%
  CER     : 8.77%


## Cell 9 — Per-sample Error Analysis

In [12]:
import pandas as pd
from jiwer import wer as single_wer

# Compute per-sample WER and sort worst → best
rows = []
for e in sample_log:
    try:
        s_wer = single_wer(e['reference'], e['hypothesis'])
    except Exception:
        s_wer = float('nan')
    rows.append({**e, 'sample_wer': round(s_wer, 4)})

df = pd.DataFrame(rows)
df_sorted = df.sort_values('sample_wer', ascending=False)

print('\n── Top-10 worst predictions ──')
display(df_sorted.head(10))

print('\n── Top-10 best predictions ──')
display(df_sorted.tail(10).iloc[::-1])


── Top-10 worst predictions ──


,reference,hypothesis,sample_wer
2331,ఓకే <unintelligible> ఉంటున్నా ఉంటున్నాం,ఓకే ఇస్తున్నామండి అన్ని పర్సరెంట్ రా సరే ఇంకొస...,2.25
508,గుడ్లగూబలు,గుర్ల గొప్పలు,2.00
2067,ముగ్గురమన్నా,ముగ్గురు మన్నా,2.00
2488,ఎంతవుతుంది,ఎంత అవుతుంది,2.00
169,అలాగేనండి,అలాగే అండి,2.00
1014,అవునండి,అవును సార్,2.00
2096,ఓకే,మెట్లు ఉన్నాయి,2.00
1926,ఎప్పుడొస్తావమ్మ,ఎప్పుడొస్తావా అమ్మ,2.00
98,ఓకే,థ్యాంక్ యూ,2.00
2425,ఆధారాలంటే,ఆధారాలు అంటే,2.00



── Top-10 best predictions ──


,reference,hypothesis,sample_wer
637,ఓకే అండి,ఓకే అండి,0.0
2679,అవునండి,అవునండి,0.0
2268,ప్రేరేపించే అవకాశాలు ఉంటాయి మంచి మానవ సంబంధాలు...,ప్రేరేపించే అవకాశాలు ఉంటాయి మంచి మానవ సంబంధాలు...,0.0
2677,హలో,హలో,0.0
2675,ఆ,ఆ,0.0
2674,ఓకే,ఓకే,0.0
2670,ఓకే,ఓకే,0.0
2669,ఓకే ఎక్కడ జరిగింది,ఓకే ఎక్కడ జరిగింది,0.0
2668,అవునండి,అవునండి,0.0
2633,ఒకటి మంచి హాజరు తెలంగాణలో ఉద్యోగస్తులు సాధారణం...,ఒకటి మంచి హాజరు తెలంగాణలో ఉద్యోగస్తులు సాధారణం...,0.0


## Cell 10 — Save Metrics & Output Files

In [13]:
import json

metrics = {
    'model'      : MODEL_ID,
    'dataset'    : DATASET_ID,
    'config'     : LANG_CONFIG,
    'split'      : SPLIT,
    'num_samples': len(references),
    'WER'        : round(word_error_rate, 6),
    'CER'        : round(char_error_rate, 6),
}

with open(METRICS_FILE, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Save full per-sample CSV for offline analysis
csv_path = os.path.join(OUTPUT_DIR, 'predictions.csv')
df.to_csv(csv_path, index=False)

print(f'Saved: {METRICS_FILE}')
print(f'Saved: {PREDICTIONS_FILE}')
print(f'Saved: {csv_path}')
print()
print('All output files are in /kaggle/working/eval_results/')
print('Download them via the Output panel on the right →')
print()
print(json.dumps(metrics, indent=2))

Saved: /kaggle/working/eval_results/metrics.json
Saved: /kaggle/working/eval_results/predictions.jsonl
Saved: /kaggle/working/eval_results/predictions.csv

All output files are in /kaggle/working/eval_results/
Download them via the Output panel on the right →

{
  "model": "ARTPARK-IISc/Vaani-FastConformer-Telugu",
  "dataset": "ai4bharat/IndicVoices",
  "config": "telugu",
  "split": "valid",
  "num_samples": 3295,
  "WER": 0.273146,
  "CER": 0.087692
}
